# Logistic Regression

In [1]:
# Basic Imports
import sys
import pandas as pd
import time
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Importing shared functions
from src.preprocessing import preprocess
from src.vectorization import vectorize
from src.evaluation import evaluate
from src.experiments import run_experiment

# Model-specific imports
from sklearn.linear_model import LogisticRegression

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df_raw = pd.read_csv("../data/Dataset.csv")

## Model 1 

Model - Logistic Regression     
Vectorization - TF-IDF with N-grams             

### Baseline:

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 43.2s   
* Training Time : 2.5s                

Metrics -           
* Accuracy:  0.96782
* Precision:  0.95957
* Recall:  0.97820
* F1:  0.96779
* Confusion Matrix:         
 [11148     501        -> False Positives            
   265      11891]     -> False Negatives 

In [3]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df_raw, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method = "tfidf")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 45.9035 seconds.
Total features =  477076


In [4]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 2.6620 seconds.


In [5]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Importance
352973             reuters  -19.644100
361547                said  -18.528369
1810             breitbart  -14.876882
432972              trumps   -9.132123
454520  washington reuters   -7.019568

Top 5 Positive Features:
         Feature  Importance
15125      video   10.903366
446457       via    8.507856
1777    breaking    6.366823
205873     image    6.172324
197585   hillary    5.396140


In [6]:
# Storing results
results = []

results.append({
    "Experiment": "Baseline",
    "Vectorizer": "tfidf",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

# Experiments

In [7]:
model = LogisticRegression(random_state=42)

### Experiment 1 (Unigram vs Unigram+Bigram)

In [8]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 16.5649 seconds.
Total features =  70042
Training completed in 0.6292 seconds.
Accuracy:  0.9671917664356228
Precision:  0.9603399433427762
Recall:  0.976061204343534
F1:  0.9671628837188619

Confusion Matrix:
 [[11159   490]
 [  291 11865]]

Top 5 Negative Features:
         Feature  Importance
54379    reuters  -22.127018
55638       said  -16.541791
1202   breitbart  -14.349670
9395        york   -9.069315
63968     trumps   -8.879073

Top 5 Positive Features:
        Feature  Importance
9014      video   10.279871
66079       via    9.882855
36281     image    6.699478
1196   breaking    6.195092
46844   october    5.642485


### Experiment 2 (Stopwords ON vs OFF)

In [9]:
results.append(
    run_experiment(
        df_raw,
        model=model, 
        vectorizer="tfidf",
        remove_stopwords=False,
        experiment_name="Kept Stopwords"
    )
)


Features created in 51.0029 seconds.
Total features =  523593
Training completed in 2.9674 seconds.
Accuracy:  0.9684940138626339
Precision:  0.9613331176185084
Recall:  0.9776242184929254
F1:  0.9684657243446477

Confusion Matrix:
 [[11171   478]
 [  272 11884]]

Top 5 Negative Features:
          Feature  Importance
388251       said  -16.435048
381305    reuters  -16.225462
2790    breitbart  -15.730893
324000         on   -7.778136
475833     trumps   -7.614857

Top 5 Positive Features:
         Feature  Importance
22112      video   11.043394
488005       via    6.784215
2753    breaking    6.225821
27321       2016    5.241862
219231   hillary    4.860810


### Experiment 3 (Title Only)

In [10]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Title Only",
        vec_text=False
    )
)


Features created in 8.9320 seconds.
Total features =  15985
Training completed in 0.0458 seconds.
Accuracy:  0.8973745011552194
Precision:  0.8860799745607759
Recall:  0.9169134583744653
F1:  0.8972176281700783

Confusion Matrix:
 [[10216  1433]
 [ 1010 11146]]

Top 5 Negative Features:
          Feature  Importance
1810    breitbart  -17.018015
15915  york times   -8.657581
11862        says   -7.381339
4836      factbox   -5.977784
15911        york   -5.789854

Top 5 Positive Features:
        Feature  Importance
15125     video   15.889239
1777   breaking    7.742029
6293    hillary    6.512331
15407     watch    6.439370
14607    tweets    4.665083


### Experiment 4 (Text Only)

In [11]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Text Only",
        vec_title=False
    )
)


Features created in 46.6716 seconds.
Total features =  461091
Training completed in 1.1280 seconds.
Accuracy:  0.9554295316110061
Precision:  0.9477762531277747
Recall:  0.9659427443237907
F1:  0.9553864200109012

Confusion Matrix:
 [[11002   647]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Importance
336988             reuters  -24.167961
345562                said  -20.436112
416987              trumps  -10.430314
438535  washington reuters   -8.664267
420072             twitter   -8.057454

Top 5 Positive Features:
          Feature  Importance
430472        via   11.603038
181600    hillary    7.371793
189888      image    7.277764
189968  image via    6.267579
269976      obama    5.814341


### Experiment 5 (Text + Title combined before vectorization)

In [12]:
df_combined = pd.DataFrame({
    'title': df_raw['title'],
    'text': df_raw['title'].fillna('') + " " + df_raw['text'].fillna(''),
    'label': df_raw['label'] 
})

results.append(
    run_experiment(
        df_combined,
        model=model,
        vectorizer="tfidf",
        experiment_name="Text and Title combined",
        vec_title=False
    )
)


Features created in 54.3577 seconds.
Total features =  471771
Training completed in 0.7342 seconds.
Accuracy:  0.9579920184835119
Precision:  0.9524659312134978
Recall:  0.9659427443237907
F1:  0.9579578135720492

Confusion Matrix:
 [[11063   586]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Importance
344318             reuters  -23.280788
353116                said  -19.821144
55807            breitbart  -11.314811
426380              trumps   -9.295632
448812  washington reuters   -8.894122

Top 5 Positive Features:
          Feature  Importance
440262        via   11.479507
440914      video   10.446018
185594    hillary    8.053941
194136      image    7.538882
194219  image via    6.420594


### Experiment 6 (min_df=2)

In [13]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="min_df=2",
        mdf=2
    )
)


Features created in 54.1900 seconds.
Total features =  550000
Training completed in 1.6729 seconds.
Accuracy:  0.9675278302877547
Precision:  0.959919191919192
Recall:  0.9772128989799276
F1:  0.9674975993019134

Confusion Matrix:
 [[11153   496]
 [  277 11879]]

Top 5 Negative Features:
           Feature  Importance
412950     reuters  -19.861736
422179        said  -18.495374
3627     breitbart  -16.025967
500749      trumps   -9.109845
49681   york times   -7.139926

Top 5 Positive Features:
         Feature  Importance
45708      video   11.071569
515119       via    8.534276
3558    breaking    6.268860
255233     image    6.075297
246305   hillary    5.324567


### Experiment 7 (min_df=10)

In [14]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="min_df=10",
        mdf=10
    )
)


Features created in 51.4188 seconds.
Total features =  195987
Training completed in 0.9371 seconds.
Accuracy:  0.9684940138626339
Precision:  0.9608112475759535
Recall:  0.9782000658111221
F1:  0.9684646014415952

Confusion Matrix:
 [[11164   485]
 [  265 11891]]

Top 5 Negative Features:
           Feature  Importance
143912     reuters  -18.955445
147508        said  -17.695253
899      breitbart  -14.498790
177675      trumps   -9.349118
7832    york times   -6.938440

Top 5 Positive Features:
         Feature  Importance
7453       video   10.383834
183475       via    8.543894
886     breaking    6.693966
84237      image    6.145000
80978    hillary    5.431574


### Experiment 8 (Removing obvious source identifiers like reuters,breitbart, washington reuters)

In [15]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 56.3310 seconds.
Total features =  475888
Training completed in 1.0182 seconds.
Accuracy:  0.95765595463138
Precision:  0.9547976501305483
Recall:  0.9626521882198091
F1:  0.9576284049980466

Confusion Matrix:
 [[11095   554]
 [  454 11702]]

Top 5 Negative Features:
                 Feature  Importance
360372              said  -19.927960
431789            trumps   -9.823138
434872           twitter   -7.824752
167629            follow   -7.080782
321424  president donald   -6.758858

Top 5 Positive Features:
         Feature  Importance
14943      video   12.045298
445271       via    9.536406
1755    breaking    7.175800
205385     image    6.990806
197102   hillary    6.085774


## Model 2

Model - Logistic Regression     
Vectorization - CountVectorizer 

### Baseline:
Features -      

* CountVectorizer  
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time :  44.2s 
* Training Time :  4.3s          

Metrics -           
* Accuracy:  0.97689
* Precision:  0.97102
* Recall:  0.98412
* F1:  0.97688
* Confusion Matrix:         
    [11292   357       -> False Positives            
     193     11963]     -> False Negatives              

In [16]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df_raw, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method="count")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 52.1672 seconds.
Total features =  477076


In [17]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 5.0277 seconds.


In [18]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9768956101659315
Precision:  0.9710227272727273
Recall:  0.9841230667982889
F1:  0.9768772385072506

Confusion Matrix:
 [[11292   357]
 [  193 11963]]

Top 5 Negative Features:
                   Feature  Importance
352973             reuters   -4.595180
1810             breitbart   -4.431182
15986                  000   -2.227315
454520  washington reuters   -1.882917
15915           york times   -1.635496

Top 5 Positive Features:
          Feature  Importance
15125       video    3.126134
446457        via    2.807345
1777     breaking    1.512943
205953  image via    1.400511
288025    october    1.178850


In [19]:
# Storing results
results.append({
    "Experiment": "Baseline",
    "Vectorizer": "count",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

## Experiments
* Removing article source names
* Varying min_df

### Experiment 1 (Unigrams Only)

In [20]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 17.7383 seconds.
Total features =  70042
Training completed in 2.4580 seconds.
Accuracy:  0.9724427641251838
Precision:  0.9664935907837092
Recall:  0.980009871668312
F1:  0.9724204578005354

Confusion Matrix:
 [[11236   413]
 [  243 11913]]

Top 5 Negative Features:
         Feature  Importance
1202   breitbart   -5.312358
54379    reuters   -5.102795
9442         000   -2.809584
9395        york   -2.499194
8530       times   -2.391555

Top 5 Positive Features:
        Feature  Importance
9014      video    3.409019
66079       via    3.328248
1196   breaking    2.273196
62619   thomson    1.358984
46844   october    1.313069


### Experiment 2 (Source phrases removal)

In [21]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 54.4516 seconds.
Total features =  475888
Training completed in 4.6983 seconds.
Accuracy:  0.9671077504725898
Precision:  0.9643936300530829
Recall:  0.9714544257979598
F1:  0.9670871193888291

Confusion Matrix:
 [[11213   436]
 [  347 11809]]

Top 5 Negative Features:
                 Feature  Importance
15784                000   -2.482234
167629            follow   -1.873979
321424  president donald   -1.859160
11706               says   -1.738267
95829                com   -1.572561

Top 5 Positive Features:
          Feature  Importance
14943       video    3.579886
445271        via    3.068323
1755     breaking    1.930871
205465  image via    1.485895
287406    october    1.364860


### Experiment 3 (varying min_df)

In [22]:
# min_df = 2
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="min_df=2",
        mdf=2
    )
)


Features created in 49.0836 seconds.
Total features =  550000
Training completed in 4.6871 seconds.
Accuracy:  0.9765595463137996
Precision:  0.9716167859466494
Recall:  0.9828068443566963
F1:  0.9765422178869341

Confusion Matrix:
 [[11300   349]
 [  209 11947]]

Top 5 Negative Features:
                   Feature  Importance
412950             reuters   -4.571596
3627             breitbart   -4.404724
50001                  000   -2.271877
523666  washington reuters   -1.914797
49681           york times   -1.627311

Top 5 Positive Features:
          Feature  Importance
45708       video    3.187474
515119        via    2.889209
3558     breaking    1.500245
255316  image via    1.460988
344255    october    1.199232


In [23]:
# min_df = 10
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="min_df=10",
        mdf=10
    )
)


Features created in 43.5883 seconds.
Total features =  195987
Training completed in 3.2470 seconds.
Accuracy:  0.9759294265910523
Precision:  0.9702801461632156
Recall:  0.9829713721618953
F1:  0.9759105710676896

Confusion Matrix:
 [[11283   366]
 [  207 11949]]

Top 5 Negative Features:
                   Feature  Importance
899              breitbart   -4.670376
143912             reuters   -4.562232
7873                   000   -2.346193
186762  washington reuters   -1.952121
7832            york times   -1.741652

Top 5 Positive Features:
          Feature  Importance
7453        video    3.144598
183475        via    2.834549
886      breaking    1.638738
84276   image via    1.416352
117977    october    1.159267


In [24]:
# min_df = 10
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="min_df=20",
        mdf=20
    )
)


Features created in 43.3004 seconds.
Total features =  87429
Training completed in 2.5885 seconds.
Accuracy:  0.9752572988867885
Precision:  0.969554274579849
Recall:  0.9823955248436986
F1:  0.9752378001814137

Confusion Matrix:
 [[11274   375]
 [  214 11942]]

Top 5 Negative Features:
                  Feature  Importance
478             breitbart   -4.799800
64009             reuters   -4.493226
4191                  000   -2.511778
83498  washington reuters   -2.062168
4170           york times   -1.813128

Top 5 Positive Features:
         Feature  Importance
3962       video    3.277061
82076        via    2.897201
472     breaking    1.744742
37872  image via    1.486212
52735    october    1.169951


In [25]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Experiment,Vectorizer,Accuracy,Precision,Recall,F1,Features,Feature Time (s),Training Time (s)
9,Baseline,count,0.976896,0.971023,0.984123,0.976877,477076,52.167171,5.027735
12,min_df=2,count,0.976560,0.971617,0.982807,0.976542,550000,49.083602,4.687069
13,min_df=10,count,0.975929,0.970280,0.982971,0.975911,195987,43.588335,3.246982
14,min_df=20,count,0.975257,0.969554,0.982396,0.975238,87429,43.300423,2.588516
10,Unigrams Only,count,0.972443,0.966494,0.980010,0.972420,70042,17.738292,2.458017
2,Kept Stopwords,tfidf,0.968494,0.961333,0.977624,0.968466,523593,51.002883,2.967405
7,min_df=10,tfidf,0.968494,0.960811,0.978200,0.968465,195987,51.418764,0.937108
0,Baseline,tfidf,0.967822,0.959571,0.978200,0.967791,477076,45.903504,2.661984
6,min_df=2,tfidf,0.967528,0.959919,0.977213,0.967498,550000,54.190025,1.672886
1,Unigrams Only,tfidf,0.967192,0.960340,0.976061,0.967163,70042,16.564888,0.629214


In [26]:
comparison_df.to_csv(
    "../results/logistic_regression.csv",
    index=False
)

## Logistic Regression Summary

### TF-IDF Baseline

Configuration:

- TF-IDF vectorization
- Separate title and text feature spaces
- Unigrams + bigrams
- Stopword removal
- `min_df=5`

Performance:

| Metric | Score |
|--------|-------|
| Accuracy | 96.78% |
| Precision | 95.96% |
| Recall | 97.82% |
| F1 Score | 96.78% |

### CountVectorizer Baseline

Configuration:

- CountVectorizer
- Separate title and text feature spaces
- Unigrams + bigrams
- Stopword removal
- `min_df=5`

Performance:

| Metric | Score |
|--------|-------|
| Accuracy | 97.69% |
| Precision | 97.10% |
| Recall | 98.41% |
| F1 Score | 97.69% |

### Key Findings

- CountVectorizer outperformed TF-IDF by approximately 0.9 percentage points in accuracy and F1 score.
- Article text contributed substantially more predictive information than titles alone.
- Keeping title and text as separate feature spaces performed better than combining them before vectorization.
- Removing stopwords had negligible impact.
- Bigrams provided a small but consistent improvement.
- Increasing `min_df` reduced feature count substantially with minimal performance loss.
- Removing source identifiers reduced performance, indicating that publisher information contributes useful signals, though models still performed well without them.
- Feature extraction remained the primary computational bottleneck across all experiments.

These results establish strong baselines for comparison with Naive Bayes, SVMs, and embedding-based approaches.